In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(12345)

In [ ]:
class Material:
    def mu(self, E):
        raise NotImplementedError


class FlatMaterial(Material):
    def __init__(self, mu0):
        self.mu0 = mu0

    def mu(self, E):
        return np.full_like(E, self.mu0)


class ResonanceMaterial(Material):
    """
    Fake neutron absorption spectrum.

    Parameters
    ----------
    baseline : float
        Constant attenuation.
    peaks : list of tuples
        (centre, width, amplitude)
    """

    def __init__(self, baseline, peaks):
        self.baseline = baseline
        self.peaks = peaks

    def mu(self, E):

        mu = np.full_like(E, self.baseline)

        for centre, width, amp in self.peaks:
            mu += amp * np.exp(
                -0.5 * ((E - centre) / width) ** 2
            )

        return mu

In [ ]:
class Rectangle:

    def __init__(self, xmin, xmax, ymin, ymax, material):
        self.xmin = xmin
        self.xmax = xmax
        self.ymin = ymin
        self.ymax = ymax
        self.material = material

    def contains(self, x, y):
        return (
            (x >= self.xmin)
            & (x <= self.xmax)
            & (y >= self.ymin)
            & (y <= self.ymax)
        )

In [ ]:
background = FlatMaterial(0.08)

fake_fe = ResonanceMaterial(
    baseline=0.08,
    peaks=[
        (1.3, 0.05, 1.5),
        (2.4, 0.12, 0.8),
        (4.8, 0.20, 0.6),
        (6.5, 0.08, 1.2),
    ],
)

insert = Rectangle(
    35,
    65,
    40,
    60,
    fake_fe,
)

In [ ]:
E = np.linspace(0.5, 8.0, 1000)

plt.figure(figsize=(8,4))
plt.plot(E, background.mu(E), label="Slab")
plt.plot(E, fake_fe.mu(E), label="Insert")
plt.xlabel("Energy (eV)")
plt.ylabel(r"$\mu(E)$")
plt.legend()
plt.show()

In [ ]:
N = 10_000_000

x = rng.uniform(0, 100, N)
y = rng.uniform(0, 100, N)

# Uniform energy for now
E = rng.uniform(0.5, 8.0, N)

In [ ]:
inside = insert.contains(x, y)

mu = background.mu(E)

mu[inside] = insert.material.mu(E[inside])

thickness = 1.0

transmission = np.exp(-mu * thickness)

keep = rng.random(N) < transmission

In [ ]:
events = pd.DataFrame(
    {
        "x": x[keep],
        "y": y[keep],
        "E": E[keep],
    }
)

print(events.head())
print()
print(f"Detected events : {len(events):,}")

In [ ]:
plt.figure(figsize=(6,6))

plt.hist2d(
    events.x,
    events.y,
    bins=200,
)

plt.xlabel("x (mm)")
plt.ylabel("y (mm)")
plt.colorbar(label="Counts")

plt.show()

In [ ]:
inside_det = insert.contains(events.x.values,
                             events.y.values)

plt.figure(figsize=(8,4))

plt.hist(
    events.E[~inside_det],
    bins=120,
    density=True,
    histtype="step",
    label="Background",
)

plt.hist(
    events.E[inside_det],
    bins=120,
    density=True,
    histtype="step",
    label="Insert",
)

plt.xlabel("Energy (eV)")
plt.ylabel("Probability density")
plt.legend()

plt.show()

In [ ]:
from PIL import Image, ImageDraw

def make_meterial_patch(text: str, value: int) -> np.ndarray:
    img = Image.new("L", (64, 64), 0)

    draw = ImageDraw.Draw(img)
    for i in range(0, 64, 12):
        draw.text((0, i), (text + " ") * 15, fill=255)
    a = np.array(img)
    # return a, img
    # print(a.min(), a.max())
    return np.where(a > 0, value, 0).astype(int)

In [ ]:
# table = np.empty((9, 18), dtype=str)
# table[...] = ""
# table[0, 0] = "H"
# table[0, 17] = "He"
# table[3, 7] = "Fe"



# element coordinates in the periodic table
elements = {
    "H": {"loc": (0, 0), "material": ResonanceMaterial(
        baseline=0.08,
        peaks=[
            (1.3, 0.05, 1.5),
            (2.4, 0.12, 0.8),
            (4.8, 0.20, 0.6),
            (6.5, 0.08, 1.2),
        ],
    ), "value": 1},
    "He": {"loc": (0, 17), "material": ResonanceMaterial(
        baseline=0.08,
        peaks=[
            (1.3, 0.05, 1.5),
            (2.4, 0.12, 0.8),
            (4.8, 0.20, 0.6),
            (6.5, 0.08, 1.2),
        ],
    ), "value": 2},
    # "Li": (1, 0),
    # "Be": (1, 1),
    # "B": (1, 12),
    # "C": (1, 13),
    # "N": (1, 14),
    # "O": (1, 15),
    # "F": (1, 16),
    # "Ne": (1, 17),
    # "Na": (2, 0),
    # "Mg": (2, 1),
    # "Al": (2, 12),
    # "Si": (2, 13),
    # "P": (2, 14),
    # "S": (2, 15),
    # "Cl": (2, 16),
    # "Ar": (2, 17),
    # "K": (3, 0),
    # "Ca": (3, 1),
    # "Sc": (3, 2),
    # "Ti": (3, 3),
    # "V": (3, 4),
    # "Cr": (3, 5),
    # "Mn": (3, 6),
    # "Fe": (3, 7),
    # "Co": (3, 8),
    # "Ni": (3, 9),
    # "Cu": (3, 10),
    # "Zn": (3, 11),
    # "Ga": (3, 12),
    # "Ge": (3, 13),
    # "As": (3, 14),
    # "Se": (3, 15),
    # "Br": (3, 16),
    # "Kr": (3, 17),
    # "Rb": (4, 0),
    # "Sr": (4, 1),
    # "Y": (4, 2),
    # "Zr": (4, 3),
    # "Nb": (4, 4),
    # "Mo": (4, 5),
    # "Tc": (4, 6),
    # "Ru": (4, 7),
    # "Rh": (4, 8),
    # "Pd": (4, 9),
    # "Ag": (4, 10),
    # "Cd": (4, 11),
    # "In": (4, 12),
    # "Sn": (4, 13),
    # "Sb": (4, 14),
    # "Te": (4, 15),
    # "I": (4, 16),
    # "Xe": (4, 17),
    # "Cs": (5, 0),
    # "Ba": (5, 1),
    # "La": (5, 2),
    # "Hf": (5, 3),
    # "Ta": (5, 4),
    # "W": (5, 5),
    # "Re": (5, 6),
    # "Os": (5, 7),
    # "Ir": (5, 8),
    # "Pt": (5, 9),
    # "Au": (5, 10),
    # "Hg": (5, 11),
    # "Tl": (5, 12),
    # "Pb": (5, 13),
    # "Bi": (5, 14),
    # "Po": (5, 15),
    # "At": (5, 16),
    # "Rn": (5, 17),
    # "Fr": (6, 0),
    # "Ra": (6, 1),
    # "Ac": (6, 2),
    # "Rf": (6, 3),
    # "Db": (6, 4),
    # "Sg": (6, 5),
    # "Bh": (6, 6),
    # "Hs": (6, 7),
    # "Mt": (6, 8),
    # "Ds": (6, 9),
    # "Rg": (6, 10),
    # "Cn": (6, 11),
    # "Nh": (6, 12),
    # "Fl": (6, 13),
    # "Mc": (6, 14),
    # "Lv": (6, 15),
    # "Ts": (6, 16),
    # "Og": (6, 17),
    # "Ce": (7, 3),
    # "Pr": (7, 4),
    # "Nd": (7, 5),
    # "Pm": (7, 6),
    # "Sm": (7, 7),
    # "Eu": (7, 8),
    # "Gd": (7, 9),
    # "Tb": (7, 10),
    # "Dy": (7, 11),
    # "Ho": (7, 12),
    # "Er": (7, 13),
    # "Tm": (7, 14),
    # "Yb": (7, 15),
    # "Lu": (7, 16),
    # "Th": (8, 3),
    # "Pa": (8, 4),
    # "U": (8, 5),
    # "Np": (8, 6),
    # "Pu": (8, 7),
    # "Am": (8, 8),
    # "Cm": (8, 9),
    # "Bk": (8, 10),
    # "Cf": (8, 11),
    # "Es": (8, 12),
    # "Fm": (8, 13),
    # "Md": (8, 14),
    # "No": (8, 15),
    # "Lr": (8, 16),
}

element_list = list(elements.keys())

In [ ]:
dx = 64
pad = 20

nrows = 9
ncols = 18

nx = ncols * (dx + pad) + pad
ny = nrows * (dx + pad) + pad

materials = np.zeros((ny, nx), dtype=int)

for i, (element, data) in enumerate(elements.items()):
    row, col = data["loc"]
    x0 = col * (dx + pad) + pad
    y0 = row * (dx + pad) + pad

    patch = make_meterial_patch(element, value=i + 1)

    materials[y0:y0+dx, x0:x0+dx] = patch



In [ ]:
%matplotlib widget

In [ ]:
import plopp as pp

pp.plot(np.flipud(materials))

In [ ]:
# Create a mu array that maps material values to absorption coefficients
def get_mu_array(materials_array, energy):
    """
    Create an absorption coefficient array based on material indices.

    Parameters
    ----------
    materials_array : np.ndarray
        Array with integer values indicating material type (0=background, 1=H, 2=He, etc.)
    energy : float or np.ndarray
        Energy value(s) at which to evaluate mu

    Returns
    -------
    np.ndarray
        Absorption coefficient array with same shape as materials_array
    """
    mu_array = np.zeros_like(materials_array, dtype=float)

    # Background (value 0)
    mu_array[materials_array == 0] = background.mu(energy)

    # Each element
    for i, (element, data) in enumerate(elements.items(), start=1):
        mask = materials_array == i
        if mask.any():
            mu_array[mask] = data["material"].mu(energy)

    return mu_array


# Example: get mu at a specific energy
E_test = 2.0  # eV
mu = get_mu_array(materials, E_test)

print(f"mu array shape: {mu.shape}")
print(f"mu range: [{mu.min():.4f}, {mu.max():.4f}]")


In [ ]:
pp.plot(np.flipud(mu))

In [ ]:
# Generate 10M events across the periodic table
N_events = 10_000_000

# Random positions across the full periodic table
x_events = rng.uniform(0, nx, N_events)
y_events = rng.uniform(0, ny, N_events)

# Random energies
E_events = rng.uniform(0.5, 8.0, N_events)

print(f"Generated {N_events:,} events")
print(f"x range: [0, {nx}], y range: [0, {ny}]")
print(f"Energy range: [0.5, 8.0] eV")


In [ ]:
# Look up which material each event hits
# Convert continuous x, y to pixel indices
x_indices = np.clip(x_events.astype(int), 0, nx - 1)
y_indices = np.clip(y_events.astype(int), 0, ny - 1)

# Get the material ID for each event
material_ids = materials[y_indices, x_indices]

print(f"Unique materials hit: {np.unique(material_ids)}")
print(f"Events in background (0): {(material_ids == 0).sum():,}")
print(f"Events in elements: {(material_ids > 0).sum():,}")


In [ ]:
# Calculate mu for each event based on its material and energy
# Using the get_mu_from_material_id function
mu_events = get_mu_from_material_id(material_ids, E_events)

print(f"mu range: [{mu_events.min():.4f}, {mu_events.max():.4f}]")
print(f"Mean mu: {mu_events.mean():.4f}")


In [ ]:
# Calculate transmission and filter events
thickness = 1.0  # mm

transmission_events = np.exp(-mu_events * thickness)

# Keep events based on transmission probability
keep_events = rng.random(N_events) < transmission_events

print(f"Initial events: {N_events:,}")
print(f"Events reaching detector: {keep_events.sum():,}")
print(f"Transmission rate: {keep_events.sum() / N_events * 100:.2f}%")


In [ ]:
# Create DataFrame with detected events
events_periodic = pd.DataFrame(
    {
        "x": x_events[keep_events],
        "y": y_events[keep_events],
        "E": E_events[keep_events],
        "material_id": material_ids[keep_events],
    }
)

print(events_periodic.head())
print()
print(f"Detected events: {len(events_periodic):,}")
print(f"\nEvents per material:")
for i, count in events_periodic.material_id.value_counts().sort_index().items():
    if i == 0:
        print(f"  Background: {count:,}")
    else:
        element = element_list[i - 1]
        print(f"  {element}: {count:,}")


In [ ]:
# Visualize spatial distribution of detected events
plt.figure(figsize=(12, 8))

plt.hist2d(
    events_periodic.x,
    events_periodic.y,
    bins=[ncols * 4, nrows * 4],
    cmap='viridis',
)

plt.xlabel("x (pixels)")
plt.ylabel("y (pixels)")
plt.colorbar(label="Counts")
plt.title(f"Detected events across periodic table ({len(events_periodic):,} events)")

plt.show()


In [ ]:
# Compare energy distributions
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(
    events_periodic[events_periodic.material_id == 0].E,
    bins=100,
    density=True,
    histtype="step",
    label="Background",
    linewidth=2,
)
plt.hist(
    events_periodic[events_periodic.material_id > 0].E,
    bins=100,
    density=True,
    histtype="step",
    label="Elements",
    linewidth=2,
)
plt.xlabel("Energy (eV)")
plt.ylabel("Probability density")
plt.legend()
plt.title("Energy distribution: Background vs Elements")

plt.subplot(1, 2, 2)
for i, element in enumerate(element_list, start=1):
    mask = events_periodic.material_id == i
    if mask.any():
        plt.hist(
            events_periodic[mask].E,
            bins=100,
            density=True,
            histtype="step",
            label=element,
            linewidth=2,
            alpha=0.8,
        )
plt.xlabel("Energy (eV)")
plt.ylabel("Probability density")
plt.legend()
plt.title("Energy distribution by element")

plt.tight_layout()
plt.show()
